# Predicting Aggregate BCI Performance and Simplifying Features

This notebook predicts each participant's mean online accuracy across `Perf_RUN_3`–`Perf_RUN_6`. It compares regularized linear models, bagged trees, boosted trees, and (when installed) XGBoost.

Because there are only 87 participants and many candidate predictors, model selection uses **repeated nested cross-validation**. Hyperparameters are selected only inside each training fold, and performance is measured on held-out outer folds. The primary metric is RMSE in percentage points; MAE, median absolute error, R², and Spearman correlation provide complementary views.

The four run-accuracy columns are used only to construct the target and are excluded from the predictors to prevent leakage. Participant identifiers, free-text fields, and every `POST_*` variable are also excluded so the model can make a genuinely prospective prediction. Final feature selection and model-family selection occur only inside training folds; untouched outer folds compare the dummy, full-feature, and reduced-feature procedures.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import linregress, spearmanr

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import make_scorer
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RepeatedKFold,
    cross_val_predict,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression

RANDOM_STATE = 42
OUTER_SPLITS = 5
OUTER_REPEATS = 5
INNER_SPLITS = 4
PERMUTATION_REPEATS = 20

sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings(
    "ignore", message="Found unknown categories.*", category=UserWarning
)

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    XGBOOST_IMPORT_ERROR = None
except Exception as error:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = error
    error_text = str(error)
    if "libomp.dylib" in error_text:
        recovery = (
            "The macOS OpenMP runtime is missing. Run `brew install libomp`, "
            "then restart the notebook kernel."
        )
    else:
        recovery = (
            "Install or repair XGBoost with `%pip install --upgrade xgboost`, "
            "then restart the notebook kernel."
        )
    print(
        f"XGBoost is unavailable ({type(error).__name__}). {recovery} "
        "The remaining models will still run."
    )

In [ ]:
search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        root
        for root in search_roots
        if (root / "data" / "processed" / "Perfomances_cleaned.csv").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the bci_cleaning project")

performances_path = PROJECT_ROOT / "data" / "processed" / "Perfomances_cleaned.csv"
performances = pd.read_csv(performances_path, sep=";")

run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
missing_run_columns = sorted(set(run_columns) - set(performances.columns))
if missing_run_columns:
    raise ValueError(f"Missing performance columns: {missing_run_columns}")

# Require all four runs so every participant's target has the same definition.
target_name = "mean_performance_accuracy"
performances[target_name] = performances[run_columns].mean(axis=1, skipna=False)
modeling_frame = performances.loc[performances[target_name].notna()].copy()

identifier_columns = ["SUJ_ID"]
free_text_columns = ["COMMENTS", "Manual activity TXT", "PRE_Pills_TXT"]
post_session_columns = [
    column for column in modeling_frame.columns if column.startswith("POST_")
]
excluded_predictors = (
    identifier_columns
    + free_text_columns
    + post_session_columns
    + run_columns
    + [target_name]
)
feature_columns = [
    column for column in modeling_frame.columns if column not in excluded_predictors
]

# These published columns contain category codes rather than continuous quantities.
coded_category_candidates = [
    "SUJ_gender",
    "EXP_gender",
    "Vision",
    "Vision_assistance",
    "Symptoms",
    "Level of study",
    "Level_knowledge neuro",
    "Meditation practice",
    "Laterality answered",
    "Manual activity",
    "PRE_Stim_normal",
    "PRE_Tabacco",
    "PRE_Tabacco_normal",
    "PRE_Alcohol",
]
categorical_features = [
    column for column in coded_category_candidates if column in feature_columns
]
numeric_features = [
    column for column in feature_columns if column not in categorical_features
]

X = modeling_frame[feature_columns].copy()
y = modeling_frame[target_name].astype(float)

dataset_summary = pd.Series({
    "participants": len(modeling_frame),
    "candidate_features": len(feature_columns),
    "numeric_features": len(numeric_features),
    "coded_categorical_features": len(categorical_features),
    "post_session_features_excluded": len(post_session_columns),
    "target_mean_percent": y.mean(),
    "target_sd_percent": y.std(),
    "target_min_percent": y.min(),
    "target_max_percent": y.max(),
})
display(dataset_summary.to_frame("value"))

if len(modeling_frame) < 5 * OUTER_SPLITS:
    raise ValueError("Too few complete participants for the requested cross-validation")

In [ ]:
def make_preprocessor(columns):
    """Build fold-local imputation, scaling, and categorical encoding."""
    selected_numeric = [column for column in numeric_features if column in columns]
    selected_categorical = [
        column for column in categorical_features if column in columns
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "one_hot",
            OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False),
        ),
    ])

    transformers = []
    if selected_numeric:
        transformers.append(("numeric", numeric_pipeline, selected_numeric))
    if selected_categorical:
        transformers.append(
            ("categorical", categorical_pipeline, selected_categorical)
        )
    return ColumnTransformer(transformers, remainder="drop")


def make_model_pipeline(regressor, columns=feature_columns):
    return Pipeline([
        ("preprocess", make_preprocessor(columns)),
        ("model", regressor),
    ])


def spearman_score(y_true, y_pred):
    correlation = spearmanr(y_true, y_pred).statistic
    return 0.0 if np.isnan(correlation) else float(correlation)


scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "median_ae": "neg_median_absolute_error",
    "r2": "r2",
    "spearman": make_scorer(spearman_score),
}

inner_cv = KFold(
    n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
outer_cv = RepeatedKFold(
    n_splits=OUTER_SPLITS,
    n_repeats=OUTER_REPEATS,
    random_state=RANDOM_STATE,
)

In [ ]:
# Small grids keep nested CV tractable while allowing each method meaningful tuning.
model_specs = {
    "Dummy mean": {
        "model": DummyRegressor(strategy="mean"),
        "grid": {},
    },
    "Ridge": {
        "model": Ridge(),
        "grid": {"model__alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]},
    },
    "Elastic net": {
        "model": ElasticNet(max_iter=20000, random_state=RANDOM_STATE),
        "grid": {
            "model__alpha": [0.01, 0.1, 1.0],
            "model__l1_ratio": [0.2, 0.8],
        },
    },
    "Random forest": {
        "model": RandomForestRegressor(
            n_estimators=250, random_state=RANDOM_STATE, n_jobs=1
        ),
        "grid": {
            "model__max_features": ["sqrt", 0.7],
            "model__min_samples_leaf": [2, 5],
        },
    },
    "Extra trees": {
        "model": ExtraTreesRegressor(
            n_estimators=250, random_state=RANDOM_STATE, n_jobs=1
        ),
        "grid": {
            "model__max_features": ["sqrt", 0.7],
            "model__min_samples_leaf": [2, 5],
        },
    },
    "Gradient boosting": {
        "model": GradientBoostingRegressor(
            loss="huber", random_state=RANDOM_STATE
        ),
        "grid": {
            "model__n_estimators": [100, 300],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [1],
        },
    },
    "Histogram gradient boosting": {
        "model": HistGradientBoostingRegressor(
            max_iter=300, random_state=RANDOM_STATE
        ),
        "grid": {
            "model__l2_regularization": [0.0, 10.0],
            "model__min_samples_leaf": [5, 15],
        },
    },
}

if XGBOOST_AVAILABLE:
    model_specs["XGBoost"] = {
        "model": XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            n_estimators=300,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        ),
        "grid": {
            "model__max_depth": [1, 2],
            "model__reg_lambda": [1.0, 10.0],
        },
    }

candidate_searches = {}
for model_name, spec in model_specs.items():
    pipeline = make_model_pipeline(clone(spec["model"]))
    if spec["grid"]:
        candidate_searches[model_name] = GridSearchCV(
            pipeline,
            param_grid=spec["grid"],
            scoring="neg_root_mean_squared_error",
            cv=inner_cv,
            n_jobs=1,
            refit=True,
        )
    else:
        candidate_searches[model_name] = pipeline

print("Models to compare:", list(candidate_searches))

In [ ]:
comparison_rows = []
outer_results = {}

for model_name, estimator in candidate_searches.items():
    print(f"Evaluating {model_name}...")
    result = cross_validate(
        estimator,
        X,
        y,
        cv=outer_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True,
        return_estimator=True,
        error_score="raise",
    )
    outer_results[model_name] = result
    comparison_rows.append({
        "Model": model_name,
        "MAE": -result["test_mae"].mean(),
        "MAE SD": result["test_mae"].std(ddof=1),
        "RMSE": -result["test_rmse"].mean(),
        "RMSE SD": result["test_rmse"].std(ddof=1),
        "Median AE": -result["test_median_ae"].mean(),
        "R2": result["test_r2"].mean(),
        "R2 SD": result["test_r2"].std(ddof=1),
        "Spearman": result["test_spearman"].mean(),
        "Train RMSE": -result["train_rmse"].mean(),
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["RMSE", "MAE"])
    .reset_index(drop=True)
)
model_comparison.insert(0, "Rank", np.arange(1, len(model_comparison) + 1))
model_comparison["RMSE overfit gap"] = (
    model_comparison["RMSE"] - model_comparison["Train RMSE"]
)
display(model_comparison.round(3))

overall_winner = model_comparison.iloc[0]["Model"]
non_dummy_results = model_comparison.loc[
    model_comparison["Model"] != "Dummy mean"
]
best_model_name = non_dummy_results.iloc[0]["Model"]
print(f"Lowest outer-CV RMSE overall: {overall_winner}")
print(f"Model used for exploratory feature ranking: {best_model_name}")
if overall_winner == "Dummy mean":
    print(
        "Caution: no candidate model beat prediction of the training-fold mean. "
        "Treat all feature rankings as exploratory rather than predictive."
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
plot_order = model_comparison["Model"].tolist()

sns.barplot(
    data=model_comparison, y="Model", x="RMSE", order=plot_order,
    color="steelblue", ax=axes[0],
)
axes[0].set_title("Lower RMSE is better")
axes[0].set_xlabel("Held-out RMSE (percentage points)")

sns.barplot(
    data=model_comparison, y="Model", x="MAE", order=plot_order,
    color="darkorange", ax=axes[1],
)
axes[1].set_title("Lower MAE is better")
axes[1].set_xlabel("Held-out MAE (percentage points)")
axes[1].set_ylabel("")

sns.barplot(
    data=model_comparison, y="Model", x="R2", order=plot_order,
    color="seagreen", ax=axes[2],
)
axes[2].axvline(0, color="black", linewidth=1)
axes[2].set_title("Higher R² is better")
axes[2].set_xlabel("Held-out R²")
axes[2].set_ylabel("")

fig.suptitle("Repeated Nested-CV Model Comparison", fontsize=16)
plt.show()

In [ ]:
# Use each already-fitted outer-fold model on only its held-out participants.
# Permuting original columns preserves one importance value per published factor.
importance_by_fold = []
outer_splits = list(outer_cv.split(X, y))
fitted_outer_estimators = outer_results[best_model_name]["estimator"]

for fold_number, ((_, test_indices), fitted_estimator) in enumerate(
    zip(outer_splits, fitted_outer_estimators), start=1
):
    permutation = permutation_importance(
        fitted_estimator,
        X.iloc[test_indices],
        y.iloc[test_indices],
        scoring="neg_root_mean_squared_error",
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE + fold_number,
        n_jobs=-1,
    )
    importance_by_fold.append(permutation.importances_mean)

importance_matrix = np.vstack(importance_by_fold)
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Mean RMSE increase": importance_matrix.mean(axis=0),
    "Importance SD": importance_matrix.std(axis=0, ddof=1),
    "Positive in folds (%)": 100 * (importance_matrix > 0).mean(axis=0),
}).sort_values(
    ["Mean RMSE increase", "Positive in folds (%)"], ascending=False
).reset_index(drop=True)
feature_importance.insert(0, "Rank", np.arange(1, len(feature_importance) + 1))

display(feature_importance.head(10).round(3))
print(
    "Importance is the increase in held-out RMSE after shuffling one feature. "
    "Values near or below zero indicate no stable predictive contribution."
)

In [ ]:
top_importance = feature_importance.head(20).sort_values(
    "Mean RMSE increase"
)
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.barh(
    top_importance["Feature"],
    top_importance["Mean RMSE increase"],
    xerr=top_importance["Importance SD"],
    color="slateblue",
    alpha=0.85,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Increase in held-out RMSE when shuffled")
ax.set_title(f"Held-Out Permutation Importance — {best_model_name}")
plt.show()

In [ ]:
# Retrieve the lowest-performing one-third of participants and retain only the
# 15 factors ranked highest by held-out permutation importance.
factor_count = 15
if len(feature_importance) < factor_count:
    raise ValueError(
        f"Only {len(feature_importance)} ranked factors are available; "
        f"cannot select {factor_count}."
    )

top_15_factors = feature_importance.head(factor_count)["Feature"].tolist()
bottom_third_count = int(np.ceil(len(modeling_frame) / 3))

bottom_third_source = pd.concat(
    [
        modeling_frame[["SUJ_ID", target_name]],
        X[top_15_factors],
    ],
    axis=1,
)
bottom_third_source = (
    bottom_third_source
    .sort_values(
        [target_name, "SUJ_ID"],
        ascending=[True, True],
        kind="mergesort",
    )
    .head(bottom_third_count)
)

# Participant ID and aggregate accuracy remain in the index for identification;
# the dataframe itself has exactly the 15 selected factor columns.
bottom_third_factor_data = bottom_third_source.set_index(
    ["SUJ_ID", target_name]
)[top_15_factors]
bottom_third_factor_data.index.names = [
    "Participant", "Mean accuracy Runs 3–6 (%)"
]

cutoff_accuracy = bottom_third_source[target_name].max()
print(
    f"Bottom third: {len(bottom_third_factor_data)} of "
    f"{len(modeling_frame)} participants; cutoff = {cutoff_accuracy:.2f}%"
)
print("Selected factors:", top_15_factors)
display(bottom_third_factor_data)

In [ ]:
# Visualize each selected factor within the bottom-performing third and rank
# which factors have the most consistent values in this subset.
consistency_rows = []
n_columns = 3
n_rows = int(np.ceil(len(top_15_factors) / n_columns))
fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(6 * n_columns, 4.2 * n_rows),
    squeeze=False,
    constrained_layout=True,
)
axes = axes.ravel()

for ax, factor in zip(axes, top_15_factors):
    subset_values = pd.to_numeric(
        bottom_third_factor_data[factor], errors="coerce"
    ).dropna()
    full_values = pd.to_numeric(X[factor], errors="coerce").dropna()
    is_coded_category = factor in categorical_features
    missing_count = len(bottom_third_factor_data) - len(subset_values)

    if subset_values.empty:
        ax.text(0.5, 0.5, "All values missing", ha="center", va="center")
        ax.set_title(factor)
        ax.set_axis_off()
        consistency_rows.append({
            "Feature": factor,
            "Type": "categorical" if is_coded_category else "numeric",
            "N": 0,
            "Missing": missing_count,
            "Median or mode": np.nan,
            "IQR": np.nan,
            "IQR / full range": np.nan,
            "Mode share": np.nan,
            "Consistency score": np.nan,
        })
        continue

    if is_coded_category:
        category_counts = subset_values.value_counts().sort_index()
        ax.bar(
            category_counts.index.astype(str),
            category_counts.values,
            color="darkorange",
            alpha=0.85,
        )
        mode_value = category_counts.idxmax()
        mode_share = category_counts.max() / len(subset_values)
        consistency_score = mode_share
        iqr = np.nan
        normalized_iqr = np.nan
        center_value = mode_value
        annotation = (
            f"mode={mode_value:g}\nmode share={mode_share:.0%}\n"
            f"n={len(subset_values)}"
        )
        ax.set_xlabel("Category code")
    else:
        discrete_numeric = subset_values.nunique() <= 10
        sns.histplot(
            subset_values,
            bins="auto",
            discrete=discrete_numeric,
            kde=(subset_values.nunique() > 4 and not discrete_numeric),
            color="steelblue",
            alpha=0.8,
            ax=ax,
        )
        first_quartile, third_quartile = subset_values.quantile([0.25, 0.75])
        iqr = third_quartile - first_quartile
        full_range = full_values.max() - full_values.min()
        normalized_iqr = iqr / full_range if full_range > 0 else 0.0
        consistency_score = float(np.clip(1 - normalized_iqr, 0, 1))
        mode_share = np.nan
        center_value = subset_values.median()
        annotation = (
            f"median={center_value:.2f}\nIQR={iqr:.2f}\n"
            f"IQR/full range={normalized_iqr:.2f}\nn={len(subset_values)}"
        )
        ax.set_xlabel("Score")

    ax.set_title(factor)
    ax.set_ylabel("Participants")
    ax.text(
        0.97,
        0.97,
        annotation,
        transform=ax.transAxes,
        ha="right",
        va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
    )
    consistency_rows.append({
        "Feature": factor,
        "Type": "categorical" if is_coded_category else "numeric",
        "N": len(subset_values),
        "Missing": missing_count,
        "Median or mode": center_value,
        "IQR": iqr,
        "IQR / full range": normalized_iqr,
        "Mode share": mode_share,
        "Consistency score": consistency_score,
    })

for unused_ax in axes[len(top_15_factors):]:
    unused_ax.remove()

fig.suptitle(
    "Selected-Factor Distributions Among the Lowest-Accuracy Participants",
    fontsize=16,
)
plt.show()

bottom_third_consistency = (
    pd.DataFrame(consistency_rows)
    .sort_values("Consistency score", ascending=False, na_position="last")
    .reset_index(drop=True)
)
bottom_third_consistency.insert(
    0, "Consistency rank", np.arange(1, len(bottom_third_consistency) + 1)
)
display(bottom_third_consistency.round(3))

rank_plot = bottom_third_consistency.dropna(
    subset=["Consistency score"]
).sort_values("Consistency score")
fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
colors = [
    "darkorange" if factor_type == "categorical" else "steelblue"
    for factor_type in rank_plot["Type"]
]
ax.barh(rank_plot["Feature"], rank_plot["Consistency score"], color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel("Consistency score (higher = more concentrated)")
ax.set_title("Factor Consistency Within the Bottom-Accuracy Third")
plt.show()

print(
    "Numeric consistency = 1 − (bottom-third IQR / full-cohort observed range). "
    "Categorical consistency = modal-category share. Compare rankings within "
    "factor type most directly; consistency does not imply predictive importance."
)

In [ ]:
# Fully nested prospective feature selection. Each outer test fold remains
# untouched while its training fold chooses the model family, hyperparameters,
# and number of transformed predictors using inner cross-validation.
SELECTION_OUTER_REPEATS = 3
SELECTION_K_VALUES = [5, 7, 10, 15]
STABILITY_THRESHOLD = 0.60
selection_outer_cv = RepeatedKFold(
    n_splits=OUTER_SPLITS,
    n_repeats=SELECTION_OUTER_REPEATS,
    random_state=RANDOM_STATE + 500,
)


def mutual_information_score(X_values, y_values):
    return mutual_info_regression(
        X_values, y_values, random_state=RANDOM_STATE
    )


def procedure_param_grids(include_selector):
    grids = []
    for model_name, spec in model_specs.items():
        if model_name == "Dummy mean":
            continue
        grid = {"model": [clone(spec["model"])]}
        grid.update(spec["grid"])
        if include_selector:
            grid["select__k"] = SELECTION_K_VALUES
        grids.append(grid)
    return grids


full_procedure = Pipeline([
    ("preprocess", make_preprocessor(feature_columns)),
    ("model", Ridge()),
])
full_procedure_search = GridSearchCV(
    full_procedure,
    param_grid=procedure_param_grids(include_selector=False),
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    n_jobs=1,
    refit=True,
)

reduced_procedure = Pipeline([
    ("preprocess", make_preprocessor(feature_columns)),
    ("select", SelectKBest(score_func=mutual_information_score, k=7)),
    ("model", Ridge()),
])
reduced_procedure_search = GridSearchCV(
    reduced_procedure,
    param_grid=procedure_param_grids(include_selector=True),
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    n_jobs=1,
    refit=True,
)

dummy_procedure = make_model_pipeline(DummyRegressor(strategy="mean"))
print("Evaluating prospective dummy procedure...")
dummy_procedure_result = cross_validate(
    dummy_procedure,
    X,
    y,
    cv=selection_outer_cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise",
)
print("Evaluating nested full-feature procedure...")
full_procedure_result = cross_validate(
    full_procedure_search,
    X,
    y,
    cv=selection_outer_cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise",
)
print("Evaluating nested reduced-feature procedure...")
reduced_procedure_result = cross_validate(
    reduced_procedure_search,
    X,
    y,
    cv=selection_outer_cv,
    scoring=scoring,
    n_jobs=-1,
    return_estimator=True,
    error_score="raise",
)


def summarize_procedure(name, result):
    return {
        "Procedure": name,
        "MAE": -result["test_mae"].mean(),
        "MAE SD": result["test_mae"].std(ddof=1),
        "RMSE": -result["test_rmse"].mean(),
        "RMSE SD": result["test_rmse"].std(ddof=1),
        "Median AE": -result["test_median_ae"].mean(),
        "R2": result["test_r2"].mean(),
        "Spearman": result["test_spearman"].mean(),
    }


prospective_procedure_comparison = pd.DataFrame([
    summarize_procedure("Dummy mean", dummy_procedure_result),
    summarize_procedure("Nested model choice — all factors", full_procedure_result),
    summarize_procedure(
        "Nested model + feature-count choice", reduced_procedure_result
    ),
]).sort_values("RMSE").reset_index(drop=True)
display(prospective_procedure_comparison.round(3))

# Recover the original published factors selected in each outer training fold.
def original_factor_name(transformed_name):
    raw_name = transformed_name.split("__", 1)[-1]
    if raw_name.startswith("missingindicator_"):
        raw_name = raw_name.removeprefix("missingindicator_")
    for factor in sorted(feature_columns, key=len, reverse=True):
        if raw_name == factor or raw_name.startswith(f"{factor}_"):
            return factor
    return None


selection_records = []
model_choice_rows = []
for fold_number, fitted_search in enumerate(
    reduced_procedure_result["estimator"], start=1
):
    fitted_pipeline = fitted_search.best_estimator_
    transformed_names = fitted_pipeline.named_steps[
        "preprocess"
    ].get_feature_names_out()
    selected_names = transformed_names[
        fitted_pipeline.named_steps["select"].get_support()
    ]
    selected_original = sorted({
        original_factor_name(name) for name in selected_names
    } - {None})
    for factor in selected_original:
        selection_records.append({"Outer fold": fold_number, "Feature": factor})
    model_choice_rows.append({
        "Outer fold": fold_number,
        "Selected model": type(
            fitted_pipeline.named_steps["model"]
        ).__name__,
        "Transformed predictors selected": int(
            fitted_pipeline.named_steps["select"].get_support().sum()
        ),
        "Original factors represented": len(selected_original),
    })

selection_record_frame = pd.DataFrame(selection_records)
selection_counts = (
    selection_record_frame.groupby("Feature")["Outer fold"]
    .nunique()
    .rename("Folds selected")
)
selection_stability = (
    pd.DataFrame({"Feature": feature_columns})
    .merge(selection_counts, on="Feature", how="left")
    .fillna({"Folds selected": 0})
)
selection_stability["Folds selected"] = selection_stability[
    "Folds selected"
].astype(int)
selection_stability["Selection frequency"] = (
    selection_stability["Folds selected"]
    / len(reduced_procedure_result["estimator"])
)
selection_stability = selection_stability.sort_values(
    ["Selection frequency", "Feature"], ascending=[False, True]
).reset_index(drop=True)

stable_features = selection_stability.loc[
    selection_stability["Selection frequency"] >= STABILITY_THRESHOLD,
    "Feature",
].tolist()
if len(stable_features) < 5:
    stable_features = selection_stability.head(5)["Feature"].tolist()
recommended_features = stable_features[:15]
recommended_count = len(recommended_features)

display(selection_stability.head(20).round(3))
display(pd.DataFrame(model_choice_rows))
display(pd.DataFrame({"Final candidate factor": recommended_features}))

stability_plot = selection_stability.head(20).sort_values(
    "Selection frequency"
)
fig, axes = plt.subplots(1, 2, figsize=(17, 7), constrained_layout=True)
axes[0].barh(
    stability_plot["Feature"], stability_plot["Selection frequency"],
    color="steelblue",
)
axes[0].axvline(STABILITY_THRESHOLD, color="crimson", linestyle="--")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Fraction of outer training folds selecting factor")
axes[0].set_title("Prospective Feature-Selection Stability")

comparison_long = prospective_procedure_comparison.melt(
    id_vars=["Procedure"],
    value_vars=["RMSE", "MAE"],
    var_name="Metric",
    value_name="Error",
)
sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Error",
    hue="Procedure",
    ax=axes[1],
)
axes[1].set_ylabel("Untouched outer-fold error (percentage points)")
axes[1].set_title("Prospective Procedure Comparison")
plt.show()

# Fit a final candidate model on all available participants. This fit is for
# future prediction only; its performance must be measured in a new cohort.
final_candidate_pipeline = make_model_pipeline(
    Ridge(), columns=recommended_features
)
final_candidate_search = GridSearchCV(
    final_candidate_pipeline,
    param_grid=procedure_param_grids(include_selector=False),
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
)
final_candidate_search.fit(X[recommended_features], y)
tuned_model = clone(
    final_candidate_search.best_estimator_.named_steps["model"]
)
print("Final candidate parameters:", final_candidate_search.best_params_)
print(
    "No independent cohort is present in this repository. Freeze this factor "
    "list and evaluate it once on new participants before making confirmatory "
    "claims. The unbiased result above applies to the nested selection procedure, "
    "not to this final refit."
)

In [ ]:
# Plot a separate univariate linear regression for every recommended factor.
# These plots describe marginal associations; they do not show each factor's
# adjusted effect in the multivariable predictive model.
if not recommended_features:
    raise ValueError("No recommended features are available to plot")

regression_rows = []
n_columns = min(3, len(recommended_features))
n_rows = int(np.ceil(len(recommended_features) / n_columns))
fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(6 * n_columns, 4.8 * n_rows),
    squeeze=False,
    constrained_layout=True,
)
axes = axes.ravel()

for ax, factor in zip(axes, recommended_features):
    factor_data = pd.DataFrame({
        "factor": pd.to_numeric(X[factor], errors="coerce"),
        "mean_performance": y,
    }).dropna()
    if factor_data["factor"].nunique() < 2:
        ax.text(0.5, 0.5, "Insufficient variation", ha="center", va="center")
        ax.set_title(factor)
        ax.set_axis_off()
        continue

    fit = linregress(
        factor_data["factor"], factor_data["mean_performance"]
    )
    is_coded_category = factor in categorical_features
    sns.regplot(
        data=factor_data,
        x="factor",
        y="mean_performance",
        x_jitter=0.08 if is_coded_category else None,
        ci=95,
        scatter_kws={"alpha": 0.65, "s": 35},
        line_kws={"color": "crimson", "linewidth": 2},
        ax=ax,
    )
    category_note = " [coded category]" if is_coded_category else ""
    ax.set_title(f"{factor}{category_note}")
    ax.set_xlabel(factor)
    ax.set_ylabel("Mean accuracy across Runs 3–6 (%)")
    ax.text(
        0.03,
        0.97,
        f"slope={fit.slope:.2f}\nR²={fit.rvalue ** 2:.3f}\n"
        f"p={fit.pvalue:.3g}, n={len(factor_data)}",
        transform=ax.transAxes,
        va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )
    regression_rows.append({
        "Feature": factor,
        "Coded categorical": is_coded_category,
        "N": len(factor_data),
        "Slope": fit.slope,
        "Intercept": fit.intercept,
        "R2": fit.rvalue ** 2,
        "Pearson r": fit.rvalue,
        "P value": fit.pvalue,
        "Slope SE": fit.stderr,
    })

for unused_ax in axes[len(recommended_features):]:
    unused_ax.remove()

fig.suptitle(
    "Univariate Associations Between Selected Factors and Mean BCI Accuracy",
    fontsize=16,
)
plt.show()

selected_feature_regressions = pd.DataFrame(regression_rows).sort_values(
    "R2", ascending=False
)
display(selected_feature_regressions.round(4))
if selected_feature_regressions["Coded categorical"].any():
    print(
        "Caution: slopes for coded categorical factors depend on their numeric "
        "codes and should not be interpreted as continuous dose-response effects."
    )

In [ ]:
# A single shuffled five-fold pass gives one out-of-fold prediction per participant
# for an interpretable observed-versus-predicted diagnostic plot.
diagnostic_columns = recommended_features
diagnostic_pipeline = make_model_pipeline(
    clone(tuned_model), columns=diagnostic_columns
)
diagnostic_cv = KFold(
    n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE + 100
)
out_of_fold_prediction = cross_val_predict(
    diagnostic_pipeline,
    X[diagnostic_columns],
    y,
    cv=diagnostic_cv,
    n_jobs=-1,
)
diagnostic_frame = pd.DataFrame({
    "Observed accuracy": y,
    "Predicted accuracy": out_of_fold_prediction,
})
diagnostic_frame["Residual"] = (
    diagnostic_frame["Observed accuracy"]
    - diagnostic_frame["Predicted accuracy"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
sns.scatterplot(
    data=diagnostic_frame,
    x="Observed accuracy",
    y="Predicted accuracy",
    ax=axes[0],
)
identity_min = min(
    diagnostic_frame["Observed accuracy"].min(),
    diagnostic_frame["Predicted accuracy"].min(),
)
identity_max = max(
    diagnostic_frame["Observed accuracy"].max(),
    diagnostic_frame["Predicted accuracy"].max(),
)
axes[0].plot(
    [identity_min, identity_max], [identity_min, identity_max], "k--"
)
axes[0].set_title("Out-of-fold predictions")

sns.scatterplot(
    data=diagnostic_frame,
    x="Predicted accuracy",
    y="Residual",
    ax=axes[1],
)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("Residuals")
for ax in axes:
    ax.set_xlabel(ax.get_xlabel() + " (%)")
plt.show()

## Interpreting the results

- Prefer the lowest held-out RMSE and MAE, but require R² and Spearman correlation to support the same conclusion.
- Compare every method with `Dummy mean`. If a model cannot beat the dummy baseline, the available factors do not yet demonstrate useful out-of-sample prediction.
- A high train-to-test RMSE gap indicates overfitting.
- The prospective procedure excludes every `POST_*` field and performs model and feature-count selection entirely inside each outer training fold.
- A candidate factor is more credible when it is selected in at least 60% of outer training folds; correlated factors can substitute for one another and reduce individual stability.
- The outer-fold scores estimate the entire nested selection procedure, not the final all-data refit. The final candidate factor list must be frozen and evaluated once on a genuinely new participant cohort.
- With fewer than 100 participants, all conclusions remain exploratory until that external validation is completed.